# RLHF on Kaggle — run once and walk away

**Fire-and-forget.** No monitoring, no per-cell editing.

1. **Settings** (right sidebar): Accelerator = **GPU T4 ×2** (or P100), Internet = **On**.
2. (Optional) tweak `MODEL` / `PRESET` in the **config** cell.
3. **Save Version ▸ Save & Run All (Commit)** → close the tab.  *(Or run headless from your
   terminal with the Kaggle API — see `kernel-metadata.json` + `scripts/run_on_kaggle.sh` in the repo.)*

It runs SFT → Reward Model → PPO → evaluation and saves checkpoints + a **`RESULTS.md`**
(reward-model accuracy, PPO-vs-SFT win-rate, sample completions) to the output.

> **Free-GPU note:** a `full` 0.5B run is ~3–5 h and free GPUs can be *preempted* (Kaggle restarts
> the run from scratch). If it keeps restarting, use `PRESET='fast'`, or split SFT/RM/PPO into
> separate commits chained via Kaggle Datasets.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable the accelerator!')
!pip install -q -U 'transformers>=4.44' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest

In [ ]:
import os
REPO_URL = 'https://github.com/TheYellowDuck/RLHF-pipeline.git'   # public repo (clone needs no token)
ATTACHED = '/kaggle/input/rlhf-pipeline'                          # fallback: attached Dataset
if REPO_URL:
    !rm -rf /kaggle/working/rlhf-pipeline && git clone -q $REPO_URL /kaggle/working/rlhf-pipeline
elif os.path.exists(ATTACHED):
    !rm -rf /kaggle/working/rlhf-pipeline && cp -r $ATTACHED /kaggle/working/rlhf-pipeline
else:
    raise SystemExit('Set REPO_URL or attach the repo as a Kaggle Dataset.')
%cd /kaggle/working/rlhf-pipeline

In [ ]:
# Pre-flight: proves the pipeline is sound on this box (tiny model, ~1 min). Aborts the run if red.
!python scripts/smoke_test.py && python -m pytest tests/ -q

## Config — the only knobs

In [ ]:
MODEL  = 'Qwen/Qwen2.5-0.5B'   # coherent text + room for PPO to improve. Smaller/faster: 'EleutherAI/pythia-160m'
PRESET = 'full'                # 'fast' (~2 h)  or  'full' (~3-5 h on 0.5B; cleaner RM + curves)

if PRESET == 'fast':
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 4000, 8000, 1024, 32
    ROLLOUT, MINI = 16, 4
else:
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 8000, 20000, 2048, 40
    ROLLOUT, MINI = 16, 4
PPO_LR = '5e-7'                # gentler than the 1e-6 default (we saw high clipfrac at 0.5B)
print(f'model={MODEL}  preset={PRESET}  sft={SFT_SAMPLES}  rm={RM_SAMPLES}  '
      f'ppo_ep={PPO_EPISODES}  rollout={ROLLOUT}  mini={MINI}  max_new={MAX_NEW}  ppo_lr={PPO_LR}')

## 1. SFT

In [ ]:
!python scripts/train_sft.py \
  -o model.name_or_path=$MODEL -o data.max_samples=$SFT_SAMPLES \
  -o train.batch_size=8 -o train.bf16=true -o output_dir=checkpoints/sft

## 2. Reward model  (initialized from the SFT checkpoint — InstructGPT recipe, higher accuracy)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=checkpoints/sft -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=500 \
  -o train.batch_size=8 -o train.bf16=true -o output_dir=checkpoints/reward_model

## 3. PPO (RL against the reward model)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=checkpoints/sft \
  -o reward_model.name_or_path=checkpoints/reward_model \
  -o ppo.total_episodes=$PPO_EPISODES -o ppo.rollout_batch_size=$ROLLOUT -o ppo.mini_batch_size=$MINI \
  -o ppo.lr=$PPO_LR -o ppo.normalize_rewards=true \
  -o generation.max_new_tokens=$MAX_NEW -o data.max_prompt_length=160

## 4. Results → written to RESULTS.md (no API key needed)

In [ ]:
import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    o = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(o.stdout)
    if o.returncode: print(o.stderr[-1500:])
    return o.stdout

rm  = run('python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --max-samples 500')
win = run(f'python scripts/evaluate.py score-policy --policy checkpoints/ppo --reward-model checkpoints/reward_model --compare checkpoints/sft --num 100 --max-new-tokens {MAX_NEW}')

md = (f'# RLHF run results\n\n- model: `{MODEL}`\n- preset: `{PRESET}`\n\n'
      f'## Reward-model accuracy (held-out HH-RLHF)\n```\n{rm}\n```\n\n'
      f'## PPO vs SFT — reward-model win-rate + sample completions\n```\n{win}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md)
print('\nSaved -> /kaggle/working/RESULTS.md')

### When it finishes
Open the completed version → **Output** → grab **`RESULTS.md`** + `rlhf-pipeline/checkpoints/`
(models + `metrics.jsonl` reward/KL curves).

- **RM accuracy** ~0.66–0.70 is the realistic target on HH-RLHF at this scale (dataset label noise caps it ~0.72).
- **OOM on the T4?** add `-o policy.use_lora=true` (PPO) / `-o model.use_lora=true` (SFT/RM), or lower `ROLLOUT`/`MINI`.
- **GRPO instead of PPO:** swap the PPO cell for `scripts/train_grpo.py` (same overrides).
- **Independent LLM-judge win-rate (optional):** add an `ANTHROPIC_API_KEY` secret, then
  `!python scripts/evaluate.py judge --policy checkpoints/ppo --base checkpoints/sft --num 100`.